### IMPORT LIBRARIES

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

### Environment Check

In [ ]:
print("Numpy version: ", np.__version__)
print("Pandas version: ", pd.__version__)

### SET PROJECT PATH

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Sample-Superstore.csv"

print("Project Root: ", PROJECT_ROOT)
print("Dataset exists: ", DATA_PATH.exists())
print("Dataset path: ", DATA_PATH)

### LOAD THE DATASET

In [ ]:
raw_df = pd.read_csv(DATA_PATH)
print(f"Raw dataset shape: {raw_df.shape}")

### Create Working Dataset

Creating a separate working copy of the raw dataset so the original data stays untouched. All cleaning will be done on the working copy.

In [ ]:
clean_df = raw_df.copy()

print("Raw shape:", raw_df.shape)
print("Working copy shape:", clean_df.shape)
print("Same object:", clean_df is raw_df)

### Convert Date Columns to Datetime

In [ ]:
clean_df['Order Date'] = pd.to_datetime(clean_df['Order Date'])
clean_df['Ship Date'] = pd.to_datetime(clean_df['Ship Date'])

## 1. Future Order Date Investigation

Objective: Identify order records with dates later than November 15, 2026, and calculate their count and percentage of the complete dataset.


1. Identify Future-Dated Order Records

In [ ]:
reference_date = pd.Timestamp('2026-11-15')
future_orders = clean_df[clean_df['Order Date'] > reference_date]

2. Calculate Total Records and Percentage

In [ ]:
future_orders_count = len(future_orders)
print("Future order records:", future_orders_count)

future_order_pct = future_orders_count / len(clean_df) * 100
print(f"Future order percentage: {future_order_pct:.2f}%")

Observations:

Using November 15, 2026, as the reference date, the investigation identified 709 records with Order Dates later than the reference date.

These records represent 6.96% of the complete dataset.

This indicates that a portion of the dataset contains future-dated order records, which should be investigated further before using the data for business analysis.

### 2. Future Order Date Range & Distribution
Objective: Check the earliest/latest dates, total span in days, sample records, and monthly distribution of future-dated orders.

1. Earliest, Latest and Date Span

In [ ]:
earliest_future_order = future_orders['Order Date'].min()
latest_future_order = future_orders['Order Date'].max()
print("Earliest future order date:", earliest_future_order)
print("Latest future order date:",latest_future_order)

future_order_span_days = (latest_future_order - earliest_future_order).days
print("Future orders span (days):", future_order_span_days)

2. Sample Future-Dated Order Records

In [ ]:
future_orders_sample = future_orders.sample(5, random_state=42)
print(future_orders_sample[['Order ID','Order Date','Sales']])

future_order_by_month = future_orders['Order Date'].dt.month.value_counts().sort_index()
print("\nFuture order by month:")
print(future_order_by_month)

Observations:

- The earliest future Order Date is November 16, 2026, and the latest is December 30, 2026, a span of 44 days.
- November has 227 future-dated records (about 15 per day over 15 days) and December has 482 (about 16 per day over 30 days).
- The daily rate is nearly the same in both months, so future-dated records are spread evenly across the period rather than concentrated near the reference date or the year-end.
- The records form a continuous stretch, not a few isolated entries. This suggests the dates are part of the dataset's overall date range, not individual entry errors. This still needs confirming against the other years before any decision.

### 3. Future Ship Date Count & Percentage

Objective: Identify shipment records with dates later than November 15, 2026, and calculate their count and percentage of the complete dataset.

1. Identify Future-Dated Shipments

In [ ]:
future_ships = clean_df[clean_df['Ship Date'] > reference_date]

2. Calculate Total Records and Percentage

In [ ]:
future_ship_count = len(future_ships)
print("Future ship records:", future_ship_count)

future_ship_pct = future_ship_count / len(clean_df) * 100
print(f"Future ship percentage: {future_ship_pct:.2f}%")

3. Orders Placed Before But Shipped After Reference Date


In [ ]:
shipped_after = clean_df[(clean_df['Order Date'] <= reference_date) & (clean_df['Ship Date'] > reference_date)]
print("Order before, ship after:", len(shipped_after))

print(shipped_after[['Order ID', 'Order Date', 'Ship Date']].head())

**Observations:**

- The investigation identified 775 records with Ship Dates later than November 15, 2026, which is 7.60% of the complete dataset.
- Future-dated shipments (775) exceed future-dated orders (709) by 66 records.
- A separate check confirmed that exactly 66 records have an Order Date on or before November 15, 2026, but a Ship Date after it.
- The sampled records were ordered on November 10, 2026, and shipped on November 16 or 17, 2026. This matches the normal shipping time of a few days, so the extra 66 records are orders placed shortly before the reference date whose shipments fall after it.
- Every future-dated order also ships after the reference date, so the 775 future shipments are the 709 future-dated orders plus these 66 records.

These findings show that future-dated shipment records come from the same date range as the future-dated orders, not from separate or unusual entries. The decision on how to handle them will be made at the end of Mission 2.1.

### 4. Future Ship Date Range & Distribution

Objective: Investigate the date range and distribution of future-dated Ship Dates to understand how far they extend and whether they are spread across the period or concentrated on specific dates.

1. Earliest, Latest and Date Span

In [ ]:
earliest_future_ship = future_ships['Ship Date'].min()
latest_future_ship = future_ships['Ship Date'].max()
print("Earliest future ship date:", earliest_future_ship)
print("Latest future ship date:",latest_future_ship)

future_ship_span_days = (latest_future_ship - earliest_future_ship).days
print("Future ship span (days):", future_ship_span_days)

2. Sample Future-Dated ship Records

In [ ]:
future_ships_sample = future_ships.sample(5, random_state=42)
print(future_ships_sample[
    ['Order ID', 'Order Date', 'Ship Date', 'Sales']
])

future_ship_by_month = future_ships['Ship Date'].dt.month.value_counts().sort_index()
print("\nFuture ship by month:")
print(future_ship_by_month)

Observations:

- The earliest future Ship Date is November 16, 2026, and the latest is January 5, 2027, a span of 50 days.
- Future Ship Dates fall in three months: November (244 records), December (485 records) and January 2027 (46 records).
- November covers 15 days (about 16 records per day) and December covers 31 days (about 16 per day). January covers only 5 days (about 9 per day), and these are the shipments of orders placed at the very end of December.
- The sampled records were ordered in December 2026 and ship within the same future window.

## Data Cleaning
Objective: Decide which cleaning actions are necessary, based on the Day 1 investigatio

### 1. Missing Values Recheck

Objective: Verify that no missing values remain in the working dataset.

Decisions:

| Issue | Day 1 Finding | Decision |
|---|---|---|
| Missing values | 0 in all 21 columns | No treatment required |
| Exact duplicates | 0 records | No removal required |
| Repeated Order IDs | Multiple products within one order | Retain |
| Data types | Dates converted to datetime in setup | No further change |
| Quantity | 1 to 14, no invalid values | Retain |
| Discount | 0.00 to 0.80, within valid range | Retain |
| Negative profit | 1,901 loss-making order-line records | Retain, not an error |
| Statistical outliers | IQR outliers in Sales, Quantity, Discount | Retain, not errors |
| Future-dated records | 709 orders, 775 ship dates beyond Nov 15, 2026 | Retain, part of the dataset's date range |


**Conclusion:**

Based on the Day 1 investigation, no unnecessary cleaning operations were required.

- No missing values or exact duplicate records were found.
- Repeated Order IDs represent multiple order lines and were retained.
- Negative profit values represent genuine loss-making records.
- Statistical outliers were retained because they are not necessarily errors.
- Quantity and discount values fall within the observed valid ranges.
- Future-dated records were retained for further review.

No records were deleted or modified during this cleaning investigation.

The dataset is ready to proceed to feature engineering, with future-dated records documented for further review.